# 06b - Validate 06's reconstructed preprocessing against `triplets_knee`

`06_submission_inference.ipynb`'s Cells 3-5 (laterality resolution, the
laterality-based slice-order fix, VOI LUT intensity normalization) were
reconstructed from README prose, not copied from `05a`'s real (unsynced)
code -- see `06`'s Cell 0 warning. This notebook is the validation step
promised there: run the **exact same functions** (copied verbatim from
`06`, not re-derived) against the 58 real gold studies' `train_series`
DICOM, and compare the resulting triplets against `triplets_knee`'s
already-known-good `.npy` files for those same 58 studies -- the ones
`05a` (the real, validated pipeline) already produced.

No GPU needed -- this is pure preprocessing, runs fine on CPU.

**What a pass looks like:** for the great majority of the 58 studies,
`06`'s reconstructed triplet should be very close to (ideally near
byte-identical to, modulo the `zoom`/float ops) `triplets_knee`'s
triplet, **in the same channel order** (not reversed). A systematic
"reversed match" for one laterality value only (e.g. every `R` study's
best match is `06`'s channel order reversed relative to the reference)
means the laterality-reversal *direction* in `06` Cell 4 is backwards --
flip the condition (`if laterality == "R"` -> `if laterality == "L"`) and
re-run this notebook before trusting `06` for a real submission.

## Cell 1 - Imports + mount (triplets_knee + competition data)

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from pydicom.pixels import apply_voi_lut
from scipy.ndimage import zoom

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
TRIPLETS_DIR = Path("/kaggle/input/datasets/alherma7/triplets-knee")
assert RAW_DIR.exists(), f"Competition data not found at {RAW_DIR}"
assert TRIPLETS_DIR.exists(), f"Triplets dataset not found at {TRIPLETS_DIR}"

OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

train = pd.read_csv(RAW_DIR / "train.csv")
train_series = pd.read_csv(RAW_DIR / "train_series.csv")

gold_mask = train[LABEL_COLS].notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID"]].reset_index(drop=True)
assert len(gold) == 58, f"Expected 58 gold studies, found {len(gold)}"
print(f"Gold studies: {len(gold)}")

gold_sag = train_series[
    train_series["StudyInstanceUID"].isin(gold["StudyInstanceUID"])
    & (train_series["Anatomical_Plane"].str.lower() == "sagittal")
]
print(f"Sagittal series among gold: {len(gold_sag)} rows, covering {gold_sag['StudyInstanceUID'].nunique()} / 58 studies")

## Cell 2 - Preprocessing functions

**Copied verbatim from `06_submission_inference.ipynb`** (Cells 2-8, as
they stand after the `slice_spacing_mm` priority fix) -- this is
deliberate: validating a re-derived copy would only prove the copy is
self-consistent, not that `06`'s actual code is correct. Only
`series_subdir` is now parameterized to `"train_series"` for this gold
comparison instead of `06`'s `"test_series"`.

In [ ]:
def count_slices(study_id, series_id, series_subdir):
    d = RAW_DIR / series_subdir / study_id / series_id
    return len(list(d.glob("*.dcm")))


def select_sagittal_series(series_df, slice_counts):
    sagittal = series_df.copy()
    sagittal["n_slices"] = sagittal["SeriesInstanceUID"].map(slice_counts).fillna(0).astype(int)

    def _pick(group):
        fluid_sensitive = group[group["Fluid_Sensitive"] == 1]
        pool = fluid_sensitive if len(fluid_sensitive) > 0 else group
        pool = pool.sort_values(["n_slices", "SeriesInstanceUID"], ascending=[False, True])
        return pool.iloc[0]

    return sagittal.groupby("StudyInstanceUID").apply(_pick, include_groups=False)


_LR_PREFIX_RE = re.compile(r"^\s*(L|R|LT|RT)[\s.\-_]", re.IGNORECASE)
_LEFT_WORD_RE = re.compile(r"\bLEFT\b", re.IGNORECASE)
_RIGHT_WORD_RE = re.compile(r"\bRIGHT\b", re.IGNORECASE)


def resolve_laterality(ds):
    tag_value = getattr(ds, "Laterality", None)
    if tag_value in ("L", "R"):
        return tag_value

    series_desc = getattr(ds, "SeriesDescription", None)
    if not isinstance(series_desc, str) or series_desc == "DummySeriesDesc!":
        return "unknown"

    prefix_match = _LR_PREFIX_RE.match(series_desc)
    if prefix_match:
        token = prefix_match.group(1).upper()
        return "L" if token in ("L", "LT") else "R"

    has_left = bool(_LEFT_WORD_RE.search(series_desc))
    has_right = bool(_RIGHT_WORD_RE.search(series_desc))
    if has_left and not has_right:
        return "L"
    if has_right and not has_left:
        return "R"
    return "unknown"


def load_series_slices_fixed(study_id, series_id, series_subdir):
    d = RAW_DIR / series_subdir / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    records = []
    first_ds = None
    for i, f in enumerate(files):
        ds = pydicom.dcmread(f)
        if i == 0:
            first_ds = ds
        records.append((float(ds.SliceLocation), ds.pixel_array, ds))
    records.sort(key=lambda r: r[0])

    laterality = resolve_laterality(first_ds)
    if laterality == "R":
        records = list(reversed(records))

    slices = [pixels for _, pixels, _ in records]
    datasets = [ds for _, _, ds in records]
    return slices, datasets, laterality


def normalize_intensity(pixel_array, ds):
    has_window = hasattr(ds, "WindowCenter") and hasattr(ds, "WindowWidth")
    if has_window:
        try:
            windowed = apply_voi_lut(pixel_array, ds)
        except Exception:
            has_window = False
    if not has_window:
        lo, hi = np.percentile(pixel_array, [0.5, 99.5])
        windowed = np.clip(pixel_array, lo, hi)

    lo, hi = windowed.min(), windowed.max()
    if hi <= lo:
        return np.zeros_like(windowed, dtype=np.float32)
    return ((windowed - lo) / (hi - lo)).astype(np.float32)


def normalize_physical_scale(pixel_array, pixel_spacing_mm, target_mm_per_pixel):
    factor = pixel_spacing_mm / target_mm_per_pixel
    return zoom(pixel_array, factor, order=1)


def center_crop_or_pad(pixel_array, crop_px):
    h, w = pixel_array.shape
    out = np.zeros((crop_px, crop_px), dtype=pixel_array.dtype)

    src_top = max(0, (h - crop_px) // 2)
    src_left = max(0, (w - crop_px) // 2)
    src = pixel_array[src_top:src_top + crop_px, src_left:src_left + crop_px]

    dst_top = max(0, (crop_px - h) // 2)
    dst_left = max(0, (crop_px - w) // 2)
    out[dst_top:dst_top + src.shape[0], dst_left:dst_left + src.shape[1]] = src
    return out


def mm_to_slice_gap(gap_mm, spacing_mm):
    return max(1, round(gap_mm / spacing_mm))


def sample_slice_indices(n_slices, n_triplets, gap):
    if n_slices <= 0 or n_triplets <= 0:
        raise ValueError("n_slices and n_triplets must be positive")
    lo, hi = gap, n_slices - 1 - gap
    if lo > hi:
        return [n_slices // 2] * n_triplets
    if n_triplets == 1:
        return [(lo + hi) // 2]
    return [int(round(x)) for x in np.linspace(lo, hi, num=n_triplets)]


def slice_spacing_mm(study_id, series_id, series_subdir):
    d = RAW_DIR / series_subdir / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    ds0 = pydicom.dcmread(files[0], stop_before_pixels=True)
    if "SpacingBetweenSlices" in ds0:
        return float(ds0.SpacingBetweenSlices)
    if "SliceThickness" in ds0:
        return float(ds0.SliceThickness)
    locs = sorted(float(pydicom.dcmread(f, stop_before_pixels=True).SliceLocation) for f in files)
    return float(np.median(np.abs(np.diff(locs))))


TARGET_MM_PER_PIXEL = 0.35
CROP_MM = 130.0
CROP_PX = round(CROP_MM / TARGET_MM_PER_PIXEL)
GAP_MM = 4.0


def preprocess_study(study_id, selected, series_subdir):
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    slices, datasets, laterality = load_series_slices_fixed(study_id, series_id, series_subdir)

    first_ds = datasets[0]
    pixel_spacing_mm = float(first_ds.PixelSpacing[0])
    spacing_between_slices_mm = slice_spacing_mm(study_id, series_id, series_subdir)

    gap = mm_to_slice_gap(GAP_MM, spacing_between_slices_mm)
    center = sample_slice_indices(len(slices), n_triplets=1, gap=gap)[0]
    idxs = [max(0, min(len(slices) - 1, center + off)) for off in (-gap, 0, gap)]

    normalized_channels = []
    for i in idxs:
        intensity_norm = normalize_intensity(slices[i], datasets[i])
        scaled = normalize_physical_scale(intensity_norm, pixel_spacing_mm, TARGET_MM_PER_PIXEL)
        cropped = center_crop_or_pad(scaled, CROP_PX)
        normalized_channels.append(cropped)

    triplet = np.stack(normalized_channels).astype(np.float32)
    return triplet, laterality

## Cell 3 - Select series + preprocess all 58 gold studies

In [ ]:
_slice_counts = {}
for _, row in gold_sag.iterrows():
    key = row["SeriesInstanceUID"]
    _slice_counts[key] = count_slices(row["StudyInstanceUID"], key, "train_series")

selected = select_sagittal_series(gold_sag, _slice_counts)
print(f"Series selected: {len(selected)} / 58 gold studies")

candidates = {}
failures = {}
laterality_counts = {"L": 0, "R": 0, "unknown": 0}
t0 = time.time()
for study_id in selected.index:
    try:
        triplet, laterality = preprocess_study(study_id, selected, "train_series")
        candidates[study_id] = (triplet, laterality)
        laterality_counts[laterality] += 1
    except Exception as e:
        failures[study_id] = repr(e)

print(f"Preprocessed: {len(candidates)} / {len(selected)} in {time.time() - t0:.1f}s")
print(f"Failures: {len(failures)}")
for sid, err in failures.items():
    print(f"  {sid[:25]}...: {err}")
print(f"Laterality breakdown: {laterality_counts}")

## Cell 4 - Compare against `triplets_knee`

For each gold study, loads the reference `triplets_knee/{study_id}.npy`
(built by the real, validated `05a` pipeline) and computes the mean
absolute error both **direct** (channel `i` vs. reference channel `i`)
and **reversed** (channel `i` vs. reference channel `2-i`). A correct
reconstruction should show low direct MAE across almost all studies; a
laterality-direction bug would show low *reversed* MAE concentrated in
one laterality group instead.

In [ ]:
rows = []
for study_id, (triplet, laterality) in candidates.items():
    ref_path = TRIPLETS_DIR / f"{study_id}.npy"
    if not ref_path.exists():
        rows.append(dict(study_id=study_id, laterality=laterality, status="no_reference"))
        continue
    ref = np.load(ref_path)
    if ref.shape != triplet.shape:
        rows.append(dict(study_id=study_id, laterality=laterality, status=f"shape_mismatch {triplet.shape} vs {ref.shape}"))
        continue

    direct_mae = float(np.mean(np.abs(triplet - ref)))
    reversed_mae = float(np.mean(np.abs(triplet - ref[::-1])))
    rows.append(dict(
        study_id=study_id, laterality=laterality, status="ok",
        direct_mae=direct_mae, reversed_mae=reversed_mae,
        better=("direct" if direct_mae <= reversed_mae else "reversed"),
    ))

comparison = pd.DataFrame(rows)
print(f"Compared: {len(comparison)} studies ({(comparison['status'] == 'ok').sum()} with a usable reference)")
print()

ok = comparison[comparison["status"] == "ok"].copy()
print("Overall: direct vs reversed match counts")
print(ok["better"].value_counts())
print()
print("Broken down by resolved laterality:")
print(ok.groupby(["laterality", "better"]).size().unstack(fill_value=0))
print()
print("direct_mae summary (should be small, e.g. << 0.05 on a [0,1]-scaled image, for a correct reconstruction):")
print(ok["direct_mae"].describe())
print()
print("Worst 10 by direct_mae (inspect these first if the aggregate result looks bad):")
print(ok.sort_values("direct_mae", ascending=False).head(10)[["study_id", "laterality", "direct_mae", "reversed_mae", "better"]])

non_ok = comparison[comparison["status"] != "ok"]
if len(non_ok):
    print()
    print("Non-OK comparisons:")
    print(non_ok)

## Interpretation

- **Pass:** almost every study's `better` is `"direct"`, with small
  `direct_mae`. `06`'s reconstruction is trustworthy as-is.
- **Laterality direction bug:** `better == "reversed"` concentrated in
  one `laterality` value (e.g. all `R`, or all `L`) -- flip the
  condition in `06` Cell 4's `load_series_slices_fixed`
  (`if laterality == "R"` <-> `if laterality == "L"`) and re-run this
  notebook.
- **Something else wrong:** high `direct_mae` *and* high `reversed_mae`
  across the board, regardless of laterality -- points at the intensity
  normalization (Cell 5) or the spacing/gap computation instead of the
  laterality fix. Inspect the worst-`direct_mae` studies' raw DICOM tags
  (`WindowCenter`/`WindowWidth` present? `SpacingBetweenSlices`
  present?) to narrow it down.
- A handful of `unknown`-laterality studies with non-trivial MAE either
  way is expected and not a bug -- there's no "correct" direction to
  compare against for those (trained un-reversed by convention, same on
  both sides of this comparison).

Report back the printed tables (especially the direct-vs-reversed
breakdown by laterality and the `direct_mae` summary) so we can decide
whether `06` is ready for a real submission attempt.

## Cell 5 - Diagnose the moderate-MAE outliers (off-by-one-slice hypothesis)

The direct-vs-reversed result above rules out a laterality-direction bug
(0/58 reversed matches). The remaining ~10 studies with moderate
`direct_mae`, scattered across L/R/unknown rather than concentrated in
one group, look more like a **center-slice selection** discrepancy than
an intensity or laterality bug -- most likely `round()`'s banker's
rounding in `mm_to_slice_gap` landing on a different integer than
`05a`'s real (unsynced) code at a `.5` gap boundary, shifting which
slice becomes the triplet's center by one index. Test this directly: for
each of the worst studies, rebuild the triplet at `center-1` and
`center+1` and see whether either neighbor matches the reference far
better than the original center did.

In [ ]:
worst = ok.sort_values("direct_mae", ascending=False).head(10)["study_id"].tolist()

diag_rows = []
for study_id in worst:
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    slices, datasets, laterality = load_series_slices_fixed(study_id, series_id, "train_series")
    first_ds = datasets[0]
    pixel_spacing_mm = float(first_ds.PixelSpacing[0])
    spacing_mm = slice_spacing_mm(study_id, series_id, "train_series")
    gap = mm_to_slice_gap(GAP_MM, spacing_mm)
    center = sample_slice_indices(len(slices), n_triplets=1, gap=gap)[0]

    ref = np.load(TRIPLETS_DIR / f"{study_id}.npy")

    row = dict(study_id=study_id, laterality=laterality, spacing_mm=round(spacing_mm, 4),
               gap=gap, raw_gap=GAP_MM / spacing_mm, center=center, n_slices=len(slices))
    for offset, label in [(-1, "center_minus_1"), (0, "center"), (1, "center_plus_1")]:
        c = max(gap, min(len(slices) - 1 - gap, center + offset))
        idxs = [max(0, min(len(slices) - 1, c + off)) for off in (-gap, 0, gap)]
        channels = []
        for i in idxs:
            intensity_norm = normalize_intensity(slices[i], datasets[i])
            scaled = normalize_physical_scale(intensity_norm, pixel_spacing_mm, TARGET_MM_PER_PIXEL)
            channels.append(center_crop_or_pad(scaled, CROP_PX))
        candidate = np.stack(channels).astype(np.float32)
        row[label] = float(np.mean(np.abs(candidate - ref)))
    diag_rows.append(row)

diag = pd.DataFrame(diag_rows)
diag["best_center"] = diag[["center_minus_1", "center", "center_plus_1"]].idxmin(axis=1)
print(diag[["study_id", "laterality", "spacing_mm", "raw_gap", "gap", "center_minus_1", "center", "center_plus_1", "best_center"]])
print()
print("best_center value counts (if 'center' dominates, it is NOT an off-by-one slice issue):")
print(diag["best_center"].value_counts())
print()
print("raw_gap distribution (values near a .5 boundary are where round() disagreement would bite):")
print(diag["raw_gap"].apply(lambda x: x - int(x)).describe())

## Cell 6 - Diagnose intensity normalization (RescaleSlope/Intercept, PhotometricInterpretation)

Off-by-one-slice is ruled out (9/10 worst studies had `center` as the
best match). Next hypothesis: `normalize_intensity` calls
`apply_voi_lut(pixel_array, ds)` directly on the **raw** pixel array,
without first applying the modality LUT (`RescaleSlope`/
`RescaleIntercept`) -- if `WindowCenter`/`WindowWidth` are defined in
the rescaled domain (the DICOM-standard case) and a study's rescale
values aren't the identity (`slope=1, intercept=0`), windowing a
not-yet-rescaled array would genuinely mismatch. Also checking
`PhotometricInterpretation` (`MONOCHROME1` needs inversion,
`MONOCHROME2` doesn't) -- unexamined anywhere in this project's EDA so
far.

In [ ]:
diag_rows2 = []
for study_id in worst:
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    d = RAW_DIR / "train_series" / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    ds0 = pydicom.dcmread(files[len(files) // 2])  # a middle slice, not just the first
    diag_rows2.append(dict(
        study_id=study_id,
        direct_mae=ok.set_index("study_id").loc[study_id, "direct_mae"],
        PhotometricInterpretation=getattr(ds0, "PhotometricInterpretation", None),
        RescaleSlope=getattr(ds0, "RescaleSlope", None),
        RescaleIntercept=getattr(ds0, "RescaleIntercept", None),
        WindowCenter=getattr(ds0, "WindowCenter", None),
        WindowWidth=getattr(ds0, "WindowWidth", None),
        VOILUTFunction=getattr(ds0, "VOILUTFunction", None),
    ))

diag2 = pd.DataFrame(diag_rows2).sort_values("direct_mae", ascending=False)
print(diag2.to_string())
print()
print("PhotometricInterpretation value counts among worst studies:")
print(diag2["PhotometricInterpretation"].value_counts())
print()
print("RescaleSlope/Intercept non-identity count (slope != 1 or intercept != 0):")
non_identity = diag2[(diag2["RescaleSlope"].astype(float) != 1.0) | (diag2["RescaleIntercept"].astype(float) != 0.0)]
print(f"{len(non_identity)} / {len(diag2)}")
print(non_identity[["study_id", "direct_mae", "RescaleSlope", "RescaleIntercept"]] if len(non_identity) else "none")

print()
print("For comparison, same tags on 5 LOW-mae studies (should look different if either tag correlates with the mismatch):")
best5 = ok.sort_values("direct_mae").head(5)["study_id"].tolist()
low_rows = []
for study_id in best5:
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    d = RAW_DIR / "train_series" / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    ds0 = pydicom.dcmread(files[len(files) // 2])
    low_rows.append(dict(
        study_id=study_id,
        direct_mae=ok.set_index("study_id").loc[study_id, "direct_mae"],
        PhotometricInterpretation=getattr(ds0, "PhotometricInterpretation", None),
        RescaleSlope=getattr(ds0, "RescaleSlope", None),
        RescaleIntercept=getattr(ds0, "RescaleIntercept", None),
    ))
print(pd.DataFrame(low_rows).to_string())

## Cell 7 - Diagnose the gap size itself (not just the center index)

`PhotometricInterpretation` (uniformly `MONOCHROME2`) and
`RescaleSlope`/`RescaleIntercept` (absent in both the worst and the best
studies alike) don't correlate with `direct_mae` -- both hypotheses
ruled out. Cell 5 only varied the **center** slice by +/-1 while holding
`gap` fixed; it never tested whether `gap` itself is off. Sweep `gap` in
a small range around the computed value (holding `center` fixed at the
gap-appropriate midpoint each time) and see whether a different gap
matches the reference far better.

In [ ]:
gap_diag_rows = []
for study_id in worst:
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    slices, datasets, laterality = load_series_slices_fixed(study_id, series_id, "train_series")
    first_ds = datasets[0]
    pixel_spacing_mm = float(first_ds.PixelSpacing[0])
    spacing_mm = slice_spacing_mm(study_id, series_id, "train_series")
    base_gap = mm_to_slice_gap(GAP_MM, spacing_mm)

    ref = np.load(TRIPLETS_DIR / f"{study_id}.npy")

    row = dict(study_id=study_id, spacing_mm=round(spacing_mm, 4), base_gap=base_gap, n_slices=len(slices))
    for gap in range(max(1, base_gap - 2), base_gap + 3):
        center = sample_slice_indices(len(slices), n_triplets=1, gap=gap)[0]
        idxs = [max(0, min(len(slices) - 1, center + off)) for off in (-gap, 0, gap)]
        channels = []
        for i in idxs:
            intensity_norm = normalize_intensity(slices[i], datasets[i])
            scaled = normalize_physical_scale(intensity_norm, pixel_spacing_mm, TARGET_MM_PER_PIXEL)
            channels.append(center_crop_or_pad(scaled, CROP_PX))
        candidate = np.stack(channels).astype(np.float32)
        row[f"gap_{gap}"] = float(np.mean(np.abs(candidate - ref)))
    gap_diag_rows.append(row)

gap_diag = pd.DataFrame(gap_diag_rows)
gap_cols = [c for c in gap_diag.columns if c.startswith("gap_")]
gap_diag["best_gap_col"] = gap_diag[gap_cols].idxmin(axis=1)
gap_diag["best_gap_mae"] = gap_diag[gap_cols].min(axis=1)
print(gap_diag.to_string())
print()
print("Which gap wins per study (if base_gap's own column dominates, gap size is NOT the issue either):")
print(gap_diag["best_gap_col"].value_counts())

## Cell 8 - Visual check (05a's real source no longer exists on Kaggle)

Three hypotheses ruled out numerically (laterality direction is
correct; center-slice and gap-size are not the cause), and
`05a_weak_dicom_preprocess.ipynb` -- the only thing that could have
settled this by direct code diff -- is confirmed gone from Kaggle. A
numeric-only comparison against a reference implementation that no
longer exists has hit its ceiling: we can't tell whether a mismatch
means *my* reconstruction is wrong or whether `triplets_knee` itself
had something unusual for that study. A visual check is the appropriate
next step -- look at the middle (`center`) channel of both triplets for
the worst few studies and judge whether either one looks like a
reasonable, correctly-windowed sagittal knee slice (visible joint
structures, no washed-out/inverted/clipped contrast) or not.

In [ ]:
import matplotlib.pyplot as plt

worst4 = ok.sort_values("direct_mae", ascending=False).head(4)["study_id"].tolist()

fig, axes = plt.subplots(len(worst4), 3, figsize=(12, 4 * len(worst4)))
for row_idx, study_id in enumerate(worst4):
    triplet, laterality = candidates[study_id]
    ref = np.load(TRIPLETS_DIR / f"{study_id}.npy")
    diff = np.abs(triplet[1] - ref[1])

    mae = ok.set_index("study_id").loc[study_id, "direct_mae"]
    mine_title = f"mine (center)\nmean={triplet[1].mean():.3f} std={triplet[1].std():.3f}"
    ref_title = f"triplets_knee (center)\nmean={ref[1].mean():.3f} std={ref[1].std():.3f}"
    diff_title = f"|diff|\nmax={diff.max():.3f}"
    for col_idx, (img, title) in enumerate([
        (triplet[1], mine_title),
        (ref[1], ref_title),
        (diff, diff_title),
    ]):
        ax = axes[row_idx, col_idx]
        ax.imshow(img, cmap="gray" if col_idx < 2 else "hot", vmin=0, vmax=1 if col_idx < 2 else None)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    axes[row_idx, 0].set_ylabel(f"{study_id[-12:]}\n(mae={mae:.3f}, lat={laterality})", fontsize=8)

plt.tight_layout()
plt.savefig("/kaggle/working/preprocessing_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved /kaggle/working/preprocessing_comparison.png")